# Lab 37: The Voice Inside

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-37.ipynb)

**What you will do:** time your accuracy solving mental arithmetic normally versus while saying "la la la" aloud, see how badly articulatory suppression disrupts your inner voice, and test whether a small open language model's arithmetic gets any better when it is prompted to "think step by step" the way you rely on hearing yourself work through a calculation.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 37 you solved a multiplication problem in your head while noticing an inner voice
narrating the steps. Repeating the same calculation while continuously saying "la la la"
aloud — **articulatory suppression** — made the calculation dramatically harder, because the
meaningless utterance occupies the same speech-production system your inner voice depends on.
Reading comprehension was similarly disrupted under suppression, even though individual words
could still be recognised.

## Record your results

Solve several two-digit multiplication problems under two conditions: normally, and while
continuously saying "la la la" aloud. Time yourself and note whether each answer was correct.

In [ ]:
# example data: replace with your own timed attempts
trials = pd.DataFrame({
    "condition": ["normal", "normal", "normal", "suppressed", "suppressed", "suppressed"],
    "problem": ["17x13", "24x16", "19x12", "17x13", "24x16", "19x12"],
    "time_s": [14, 22, 18, 41, 55, 48],
    "correct": [True, True, True, False, True, False],
})
display(trials)

trials.groupby("condition")[["time_s"]].mean().plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3), legend=False)
plt.ylabel("mean time (s)"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 37 does not give a specific published number for how much slower or less accurate
suppressed arithmetic becomes -- it reports only that the calculation becomes "dramatically
harder, often impossible." The honest comparison here is within yourself: how much did your
own time and accuracy change between the two conditions?

In [ ]:
normal = trials[trials.condition == "normal"]
suppressed = trials[trials.condition == "suppressed"]
print(f"Normal: {normal.time_s.mean():.0f} s mean, {normal.correct.mean():.0%} correct")
print(f"Suppressed: {suppressed.time_s.mean():.0f} s mean, {suppressed.correct.mean():.0%} correct")
if suppressed.time_s.mean() > normal.time_s.mean() or suppressed.correct.mean() < normal.correct.mean():
    print("Slower and/or less accurate under suppression, matching the disruption Lab 37 describes.")
else:
    print("Little difference here -- some people learn to mouth the suppression syllable"
          " quite automatically, freeing more of the speech system than intended.")

## The AI side

This is a hands-on version of the book's chain-of-thought comparison. DistilGPT-2, a small
open language model, is given the same arithmetic problem twice: once asked for the answer
directly, and once prompted with "Let's think step by step," the technique shown to improve
reasoning in much larger language models (Wei et al., 2022). DistilGPT-2 is far too small to
reliably do arithmetic correctly either way -- that is not the point here. Look instead at
whether the step-by-step prompt makes the model generate anything resembling an externalised
intermediate working-out, the way your own inner voice narrates a calculation.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

problem = "17 x 13 ="
direct_prompt = f"Q: What is {problem.replace('=', '')}? A: The answer is"
cot_prompt = f"Q: What is {problem.replace('=', '')}? Let's think step by step."

def generate(prompt, max_new_tokens=40):
    ids = tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

results = pd.DataFrame({
    "condition": ["direct answer", "'let's think step by step'"],
    "completion": [generate(direct_prompt), generate(cot_prompt)],
})
for _, row in results.iterrows():
    print(f"--- {row['condition']} ---\n{row['completion']}\n")
print(f"(For reference, the correct answer to {problem} is {17 * 13}.)")

Do not expect DistilGPT-2 to get 17 x 13 right in either condition -- at this small scale,
chain-of-thought prompting mainly changes the shape of the output, not its correctness; the
accuracy gains the book describes are documented in much larger models. The deeper parallel
the book draws is architectural, not about getting the right answer: an externalised sequence
of intermediate tokens acts as a scratch pad that extends working capacity beyond a single
pass, in the model as in your own suppressed-versus-unsuppressed arithmetic. The mechanisms
stay different -- motor simulation and auditory prediction in you; token-by-token generation
in the model -- even where the function served looks similar.

## Pool the class data

Pool accuracy and time, with and without suppression, across the class (anonymous IDs, never
names), and look at the within-person difference rather than raw scores, since some people
are simply faster at mental arithmetic than others regardless of condition.

In [ ]:
import io
csv_text = """id,condition,time_s,correct
P01,normal,15,True
P01,suppressed,38,False
P02,normal,20,True
P02,suppressed,45,True
P03,normal,12,True
P03,suppressed,30,False
P04,normal,25,True
P04,suppressed,60,False
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
time_diff = (df[df.condition == "suppressed"].set_index("id").time_s
             - df[df.condition == "normal"].set_index("id").time_s)

boot = [time_diff.sample(len(time_diff), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean slowdown under suppression: {time_diff.mean():.0f} s "
      f"(95% bootstrap interval {lo:.0f} to {hi:.0f} s, n = {len(time_diff)})")
acc_by_condition = df.groupby("condition").correct.mean()
acc_by_condition.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3))
plt.ylabel("proportion correct"); plt.xticks(rotation=0); plt.ylim(0, 1); plt.show()

## Questions to think about

1. The book calls inner speech "prediction without action" — a simulated sound with no motor command sent to your vocal tract. Articulatory suppression occupies the motor system with "la la la." What does the size of the disruption you measured suggest about how much of mental arithmetic genuinely depends on that motor simulation, rather than on some other, non-verbal form of working memory?
2. The book describes deaf signers experiencing "inner signing" instead of inner speech. Based on the motor theory of inner speech, what would you predict articulatory suppression (blocking the vocal tract) would do to a deaf signer's mental arithmetic, compared with blocking their hands?
3. In the AI Lab, "Let's think step by step" did not make DistilGPT-2 correct. What specific feature of larger models -- as opposed to just having more of the same architecture -- might explain why the same prompt reliably helps them, according to the book's account?
4. The book asks whether an auditory inner voice is essential for step-by-step thinking, or just one biological implementation of a more general planning process that written steps can also support. Having watched a model produce written, externalised steps with no auditory experience behind them at all, where do you now stand on that question?

## Challenge

Test reading comprehension rather than arithmetic: read a short, unfamiliar paragraph aloud
while saying "la la la" under your breath, then immediately summarise it from memory. Compare
your summary's completeness with a paragraph you read normally, and note whether individual
words felt recognisable even when the overall meaning did not stick.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-37.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")